In [1]:
import os, requests

if not os.path.exists("the-verdict.txt"):
    url = (
        "https://raw.githubusercontent.com/rasbt/"
        "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
        "the-verdict.txt"
    )
    file_path = "the-verdict.txt"

    response = requests.get(url, timeout=30)
    response.raise_for_status()
    with open(file_path, "wb") as f:
        f.write(response.content)

/Users/hrithikkoduri/Desktop/HF/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

print(raw_text[:100])

I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no g


In [3]:
print("Total number of characters:", len(raw_text))

Total number of characters: 20479


## Tokenization

### Approach 1

In [4]:
import re 
def preprocess_text(text):

    # Remove punctuation
    result = re.findall(r"\w+|[^\w\s]", text)
    result  = [item.strip() for item in result if item.strip()]

    return result
    

In [5]:

text = "Hello, world! This is a test."

# Split on word boundaries and preserve special characters as separate tokens
result = preprocess_text(text)
print(result)


['Hello', ',', 'world', '!', 'This', 'is', 'a', 'test', '.']


In [6]:

preprocessed_text = preprocess_text(raw_text)
len(preprocessed_text)

4827

In [7]:
print(preprocessed_text[:50])

['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '-', '-', 'though', 'a', 'good', 'fellow', 'enough', '-', '-', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in', 'the', 'height', 'of', 'his', 'glory', ',', 'he', 'had', 'dropped', 'his', 'painting', ',', 'married', 'a', 'rich', 'widow', ',', 'and']


### Convert tokens to token Ids

Creating vocabulary

In [8]:
all_words = sorted(set(preprocessed_text))
vocab_size = len(all_words)
print("Unique tokens:", vocab_size)



Unique tokens: 1148


In [9]:
vocab = {token:integer for integer, token in enumerate(all_words)}

vocab

{'!': 0,
 '"': 1,
 "'": 2,
 '(': 3,
 ')': 4,
 ',': 5,
 '-': 6,
 '.': 7,
 ':': 8,
 ';': 9,
 '?': 10,
 'A': 11,
 'Ah': 12,
 'Among': 13,
 'And': 14,
 'Are': 15,
 'Arrt': 16,
 'As': 17,
 'At': 18,
 'Be': 19,
 'Begin': 20,
 'Burlington': 21,
 'But': 22,
 'By': 23,
 'Carlo': 24,
 'Chicago': 25,
 'Claude': 26,
 'Come': 27,
 'Croft': 28,
 'Destroyed': 29,
 'Devonshire': 30,
 'Don': 31,
 'Dubarry_': 32,
 'Emperors': 33,
 'Florence': 34,
 'For': 35,
 'Gallery': 36,
 'Gideon': 37,
 'Gisburn': 38,
 'Gisburns': 39,
 'Grafton': 40,
 'Greek': 41,
 'Grindle': 42,
 'Grindles': 43,
 'HAD': 44,
 'Had': 45,
 'Hang': 46,
 'Has': 47,
 'He': 48,
 'Her': 49,
 'Hermia': 50,
 'His': 51,
 'How': 52,
 'I': 53,
 'If': 54,
 'In': 55,
 'It': 56,
 'Jack': 57,
 'Jove': 58,
 'Just': 59,
 'Lord': 60,
 'Made': 61,
 'Miss': 62,
 'Money': 63,
 'Monte': 64,
 'Moon': 65,
 'Mr': 66,
 'Mrs': 67,
 'My': 68,
 'Never': 69,
 'No': 70,
 'Now': 71,
 'Nutley': 72,
 'Of': 73,
 'Oh': 74,
 'On': 75,
 'Once': 76,
 'Only': 77,
 'Or': 78,

In [10]:
for i, item in enumerate(vocab.items()):
    print(i, item)
    if i > 50:
        break


0 ('!', 0)
1 ('"', 1)
2 ("'", 2)
3 ('(', 3)
4 (')', 4)
5 (',', 5)
6 ('-', 6)
7 ('.', 7)
8 (':', 8)
9 (';', 9)
10 ('?', 10)
11 ('A', 11)
12 ('Ah', 12)
13 ('Among', 13)
14 ('And', 14)
15 ('Are', 15)
16 ('Arrt', 16)
17 ('As', 17)
18 ('At', 18)
19 ('Be', 19)
20 ('Begin', 20)
21 ('Burlington', 21)
22 ('But', 22)
23 ('By', 23)
24 ('Carlo', 24)
25 ('Chicago', 25)
26 ('Claude', 26)
27 ('Come', 27)
28 ('Croft', 28)
29 ('Destroyed', 29)
30 ('Devonshire', 30)
31 ('Don', 31)
32 ('Dubarry_', 32)
33 ('Emperors', 33)
34 ('Florence', 34)
35 ('For', 35)
36 ('Gallery', 36)
37 ('Gideon', 37)
38 ('Gisburn', 38)
39 ('Gisburns', 39)
40 ('Grafton', 40)
41 ('Greek', 41)
42 ('Grindle', 42)
43 ('Grindles', 43)
44 ('HAD', 44)
45 ('Had', 45)
46 ('Hang', 46)
47 ('Has', 47)
48 ('He', 48)
49 ('Her', 49)
50 ('Hermia', 50)
51 ('His', 51)


### Simple tokenizer class in python

- encode  - splits the text into tokens and carries out the **string-to-integer** mapping to produce token IDs via the vocab
- decode  - carries out the reverse **integer-to-string** mapping to covert token ids back to text

In [11]:
class SimpleTokenizerV1:
    def __init__(self, text):
        self.text = text
        self.all_words = sorted(set(preprocessed_text))
        self.vocab = {token: integer for integer, token in enumerate(self.all_words)}
        self.string_to_ids = self.vocab
        self.ids_to_string = {integer: token for token, integer in self.vocab.items()}
    
    def tokenize(self, text):
        def preprocess_text(text):
            # Remove punctuation
            result = re.findall(r"\w+|[^\w\s]", text)
            result = [item.strip() for item in result if item.strip()]
            return result
        return preprocess_text(text)
    
    def encode(self, text):
        tokenized_text = self.tokenize(text)  # splits the text into tokens
        string_to_ids = [self.string_to_ids[token] for token in tokenized_text]
        return string_to_ids
    
    def decode(self, ids):
        text = " ".join([self.ids_to_string[id] for id in ids])
        # Remove space before special chars (e.g., It' s → It's)
        text = re.sub(r"\s+([^\w\s])", r"\1", text)
        return text.strip()

In [12]:
tokenizer = SimpleTokenizerV1(raw_text)

In [13]:
print(tokenizer.encode("It's the last he painted"))

[56, 2, 867, 1006, 615, 546, 760]


In [14]:
print(tokenizer.decode(tokenizer.encode("It's the last he painted")))

It' s the last he painted


In [15]:
# extand with <|unk|> and <\endoftext\>


class SimpleTokenizerV2:
    def __init__(self, text):
        self.text = text
        self.all_words = sorted(set(preprocessed_text))
        self.all_words.extend(["<|unk|>", "<|endoftext|>"])
        self.vocab = {token: integer for integer, token in enumerate(self.all_words)}
        self.string_to_ids = self.vocab
        self.ids_to_string = {integer: token for token, integer in self.vocab.items()}
    
    def tokenize(self, text):
        def preprocess_text(text):
            # Remove punctuation
            result = re.findall(r"<\|endoftext\|>|<\|unk\|>|\w+|[^\w\s]", text)
            result = [item.strip() for item in result if item.strip()]
            result = [item if item in self.vocab else "<|unk|>" for item in result]
            return result

        return preprocess_text(text)
    
    def encode(self, text):
        tokenized_text = self.tokenize(text)  # splits the text into tokens
        string_to_ids = [self.string_to_ids[token] for token in tokenized_text]
        return string_to_ids
    
    def decode(self, ids):
        text = " ".join([self.ids_to_string[id] for id in ids])
        # Remove space before special chars (e.g., It' s → It's)
        text = re.sub(r"\s+([^\w\s])", r"\1", text)
        return text.strip()

In [16]:
tokenizer = SimpleTokenizerV2(raw_text)
print(tokenizer.encode("Hello World, This is a test"))
print(tokenizer.decode(tokenizer.encode("Hello World, This is a test")))


[1148, 1148, 5, 97, 598, 129, 1148]
<|unk|><|unk|>, This is a<|unk|>


In [17]:
text1 = "Hello, This is a test."
text2 = "We are joinging texts."
text = " <|endoftext|> ".join([text1, text2])

print(text)



Hello, This is a test. <|endoftext|> We are joinging texts.


In [18]:
print(tokenizer.encode(text))

[1148, 5, 97, 598, 129, 1148, 7, 1149, 107, 182, 1148, 1148, 7]


In [19]:
print(tokenizer.decode(tokenizer.encode(text)))

<|unk|>, This is a<|unk|>.<|endoftext|> We are<|unk|><|unk|>.


## Byte Pair Encoding (BPE)

In [20]:
from importlib.metadata import version
import tiktoken
print(version("tiktoken"))

0.14.0


In [21]:
tokenizer  =  tiktoken.get_encoding("gpt2")

In [24]:
text = (
    "Hello, would you like tea? <|endoftext|> In the sunlit terraces "
    "of someunknownplace"
)

integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
print(integers)

[15496, 11, 561, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 286, 617, 34680, 5372]


In [25]:
strings = tokenizer.decode(integers)
print(strings)

Hello, would you like tea? <|endoftext|> In the sunlit terraces of someunknownplace


In [26]:
text = "Akwirw ier"
integers = tokenizer.encode(text)
print(integers)

[33901, 86, 343, 86, 220, 959]


In [29]:
map = {item: tokenizer.decode([item]) for item in integers}
print(map)


{33901: 'Ak', 86: 'w', 343: 'ir', 220: ' ', 959: 'ier'}


In [30]:
text = "Akwirw ier"
map = {item: tokenizer.decode([item]) for item in tokenizer.encode(text)}
print(map)


{33901: 'Ak', 86: 'w', 343: 'ir', 220: ' ', 959: 'ier'}


In [32]:
print(raw_text[0:100])
print(len(raw_text))

I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no g
20479


In [33]:
encoded_text = tokenizer.encode(raw_text)
print(encoded_text[0:100])
print(len(encoded_text))


[40, 367, 2885, 1464, 1807, 3619, 402, 271, 10899, 2138, 257, 7026, 15632, 438, 2016, 257, 922, 5891, 1576, 438, 568, 340, 373, 645, 1049, 5975, 284, 502, 284, 3285, 326, 11, 287, 262, 6001, 286, 465, 13476, 11, 339, 550, 5710, 465, 12036, 11, 6405, 257, 5527, 27075, 11, 290, 4920, 2241, 287, 257, 4489, 64, 319, 262, 34686, 41976, 13, 357, 10915, 314, 2138, 1807, 340, 561, 423, 587, 10598, 393, 28537, 2014, 198, 198, 1, 464, 6001, 286, 465, 13476, 1, 438, 5562, 373, 644, 262, 1466, 1444, 340, 13, 314, 460, 3285, 9074, 13, 46606, 536]
5145


In [34]:
encoded_sample = encoded_text[50:]

One of the most intuitve way to create a input-target pair is to create two variables x and y , where x containsd the input tokens and y contains the targets which are the inputs shifted by 1

In [39]:
context_size = 4
x = encoded_sample[:context_size]
y = encoded_sample[1:context_size+1]
print(f"x: {x}")
print(f"y:      {y}")

x: [290, 4920, 2241, 287]
y:      [4920, 2241, 287, 257]


In [42]:
for i in range (1, context_size+1):
    context = encoded_sample[:i] # all the token(s) except the last one
    target = encoded_sample[i] # the last token
    print(f"context: {context} ---> target: {target}")


context: [290] ---> target: 4920
context: [290, 4920] ---> target: 2241
context: [290, 4920, 2241] ---> target: 287
context: [290, 4920, 2241, 287] ---> target: 257


In [47]:
for i in range (1, context_size+1):
    context = tokenizer.decode(encoded_sample[:i]) # all the token(s) except the last one
    target = tokenizer.decode([encoded_sample[i]]) # the last token
    print(f"{context} ---> {target}")

 and --->  established
 and established --->  himself
 and established himself --->  in
 and established himself in --->  a


### Dataloader

In [49]:
import torch
from torch.utils.data import DataLoader, Dataset

class GPTDatasetV1(Dataset):
    def __init__(self, text, tokenizer, max_length, stride):

        self.input_ids = []
        self.target_ids= []
        token_ids = tokenizer.encode(text)
        for i in range(0, len(token_ids) - max_length, stride):
            input_ids = token_ids[i:i+max_length]
            target_ids = token_ids[i+1:i+max_length+1]
            self.input_ids.append(torch.tensor(input_ids))
            self.target_ids.append(torch.tensor(target_ids))

    def __len__(self):
        return len(self.input_ids)
    
    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]
       

In [ ]:
def create_dataloader(text, tokenizer, max_length=256, stride=128, batch_size=4, shuffle=True, drop_last=True, num_workers=0):
    dataset = GPTDatasetV1(text, tokenizer, max_length, stride)
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers,
    )
    return dataloader

In [52]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

tokenizer = tiktoken.get_encoding("gpt2")

stride indicates number of positions inputs shift across batches, emulating a sliding window

In [53]:
## Lets test dataloader with a batch size of 1 and context size of 4
batch_size = 1
context_size = 4
stride = 1

dataloader = create_dataloader(raw_text, tokenizer, max_length=context_size, stride=stride, batch_size=batch_size, shuffle=False, drop_last=False, num_workers=0)
print(dataloader)

    

In [56]:
# COnvert dataloader into a python iteratoor to fetch the next entry via pythons buil in next() function
data_iter = iter(dataloader)
first_batch = next(data_iter)
print(first_batch)



[tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]


In [57]:
second_batch = next(data_iter)
print(second_batch)


[tensor([[ 367, 2885, 1464, 1807]]), tensor([[2885, 1464, 1807, 3619]])]


In [61]:
# max_lenght = 2 and stride =2 
context_size = 2
stride = 2

dataloader2 = create_dataloader(raw_text, tokenizer, max_length=context_size, stride=stride, batch_size=batch_size, shuffle=False, drop_last=False, num_workers=0)
print(dataloader2)

data_iter2 = iter(dataloader2)
first_batch2 = next(data_iter2)
print(first_batch2)

second_batch2 = next(data_iter2)
print(second_batch2)


[tensor([[ 40, 367]]), tensor([[ 367, 2885]])]
[tensor([[2885, 1464]]), tensor([[1464, 1807]])]


In [62]:
# max_lenght = 8 and stride =2 
context_size = 8
stride = 2

dataloader3 = create_dataloader(raw_text, tokenizer, max_length=context_size, stride=stride, batch_size=batch_size, shuffle=False, drop_last=False, num_workers=0)
print(dataloader3)

data_iter3 = iter(dataloader3)
first_batch3 = next(data_iter3)
print(first_batch3)

second_batch3 = next(data_iter3)
print(second_batch3)

[tensor([[  40,  367, 2885, 1464, 1807, 3619,  402,  271]]), tensor([[  367,  2885,  1464,  1807,  3619,   402,   271, 10899]])]
[tensor([[ 2885,  1464,  1807,  3619,   402,   271, 10899,  2138]]), tensor([[ 1464,  1807,  3619,   402,   271, 10899,  2138,   257]])]


When creating multiple batches from input dataset we slide an input window across the tewxt. Is stride is set to 1 we shift the input window by 1 in the next batch. If we set stride equal to input window size (context size) , we can prevent thoverlaps netween the batches as in dataloader 2

In [64]:

context_size = 4
stride = 4
batch_size = 8

dataloader = create_dataloader(raw_text, tokenizer, max_length=context_size, stride=stride, batch_size=batch_size, shuffle=False, drop_last=False, num_workers=0)
print(dataloader)

data_iter = iter(dataloader)
first_batch = next(data_iter)
first_batch_inputs, first_batch_targets = first_batch
print(f"\nfirst_batch_inputs: {first_batch_inputs}")
print(f"\nfirst_batch_targets: {first_batch_targets}")

second_batch = next(data_iter)
second_batch_inputs, second_batch_targets = second_batch
print(f"\nsecond_batch_inputs: {second_batch_inputs}")
print(f"\nsecond_batch_targets: {second_batch_targets}")



first_batch_inputs: tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

first_batch_targets: tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])

second_batch_inputs: tensor([[  287,   262,  6001,   286],
        [  465, 13476,    11,   339],
        [  550,  5710,   465, 12036],
        [   11,  6405,   257,  5527],
        [27075,    11,   290,  4920],
        [ 2241,   287,   257,  4489],
        [   64,   319,   262, 34686],
        [41976,    13,   357, 10915]])

second_batch_target

## Creating TOken Embeddings

Initialize embedding weights with random values, this initilaiztion serves as the starting point for LLM's learning process.

In [66]:
# just for an example 
vocab_size  = 6
output_dim = 3

torch.manual_seed(123)
embedding_layer = torch.nn.Embedding(vocab_size, output_dim)
print(embedding_layer.weight)


Parameter containing:
tensor([[ 0.3374, -0.1778, -0.1690],
        [ 0.9178,  1.5810,  1.3010],
        [ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096]], requires_grad=True)


In [67]:
input_ids = torch.tensor([1, 2, 3, 4])

embeddings = embedding_layer(input_ids)
print(embeddings)



tensor([[ 0.9178,  1.5810,  1.3010],
        [ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-1.1589,  0.3255, -0.6315]], grad_fn=<EmbeddingBackward0>)


- Embedding layer is esstinally a lookup that retrieves rows from embedding layers weight matrix
- embedding layer is just a more efficient approach to the one-hot-encoding and matirx multiplication approach hence can be seen as a neural network layer that can be optimized via backprop

### Positional Encoding

- Minor shortcoming of LLMs is that their self attention mechanism doesnt tkae into account the position or the order of the tokens within a sequence.
- Same token id will always get mapped to the same vector representation regardless of where the token id is positioned in the input sequence
- since self attention mechanism of LLMs is position-agnostics, it is helpful to inject additional positional information into the LLM

Two ways -
1. Absoulte positional encoding -  for each position a unique embeddings is added to the token's embeding to convery the positional information (At which exact position)
2. RElative Positional embeddings- emphasis is on the relative position or distance between the tokens, (How far apart)

In [75]:
embedding_output_dim = 256
vocab_size = 50257

token_embedding_layer = torch.nn.Embedding(vocab_size, embedding_output_dim)
print(token_embedding_layer.weight.shape)

torch.Size([50257, 256])


In [ ]:
context_size = 4
batch_size = 8
stride = 4

dataloader = create_dataloader(raw_text, tokenizer, max_length=context_size, stride=stride, batch_size=batch_size, shuffle=False, drop_last=False, num_workers=0)

data_iter = iter(dataloader)
first_batch = next(data_iter)
first_batch_inputs, first_batch_targets = first_batch
print(f"\nfirst_batch_inputs: {first_batch_inputs}")
print(f"\nfirst_batch_targets: {first_batch_targets}")



first_batch_inputs: tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

first_batch_targets: tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])


In [73]:
first_batch_token_embeddings = token_embedding_layer(first_batch_inputs)
print(f"\nfirst_batch_token_embeddings shape: {first_batch_token_embeddings.shape}")



first_batch_token_embeddings shape: torch.Size([8, 4, 256])


In [78]:
torch.arange(context_size)

tensor([0, 1, 2, 3])

In [81]:
positional_embedding_layer = torch.nn.Embedding(context_size, embedding_output_dim)
positional_embeddings = positional_embedding_layer(torch.arange(context_size))
print(positional_embedding_layer.weight.shape)


torch.Size([4, 256])


In [ ]:
print(f"first_batch_token_embeddings shape: {first_batch_token_embeddings[0].shape}")
print(f"positional_embeddings shape: {positional_embeddings.shape}")

first_batch_token_embeddings shape: torch.Size([4, 256])
positional_embeddings shape: torch.Size([4, 256])


In [85]:
first_batch_input_embeddings = first_batch_token_embeddings + positional_embeddings
print(f"first_batch_input_embeddings shape: {first_batch_input_embeddings.shape}")

first_batch_input_embeddings shape: torch.Size([8, 4, 256])
